In [1]:
"""
R2R Phase 1 — Step 1: Identify Disputed Tasks
================================================
Reads the long-format production-run ratings and splits tasks into:

  - auto_resolved.csv   — full 4/4 agreement across human/claude/openai/gemini.
                           No deliberation needed; label passes straight through.
  - disputed_tasks.csv  — everything else (3/1, 2/2, 2/1/1, 4-way splits, etc.).
                           These need the consensus round.

Dispute threshold (per your instruction):
  - 4/4 unanimous agreement → auto-resolved
  - Anything less than 4/4  → disputed, goes to consensus round

Input:
  unified_rater_long_format.csv — long format, one row per (task_id, provider, run_id):
      task_id, task_statement, tier, provider, model, temperature, run_id,
      classification, reasoning
      provider values: human | claude | openai | gemini
      tier values: human_expert | mid | flagship

Outputs (written to outputs/):
  auto_resolved.csv   — task_id, task_statement, final_label, resolution_method,
                         vote_tally_initial
  disputed_tasks.csv  — task_id, task_statement, rater, classification, reasoning
                         (one row per task_id x rater, collapsed across runs —
                         this is what you read to prepare human_consensus.csv, and what
                         02_consensus_round.py consumes directly)

Usage:
    python 01_find_disputed.py
"""

from __future__ import annotations

import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

INPUT_CSV = "june 23 unified_rater_long_format.csv"

# Map provider values in your CSV to the canonical rater keys used downstream.
PROVIDER_TO_RATER = {
    "human": "human",
    "claude": "claude",
    "openai": "openai",
    "gemini": "gemini",
}
ALL_RATERS = ["human", "claude", "openai", "gemini"]

# Only mid-tier model rows are production inputs; human rows have tier
# "human_expert" and are always kept regardless of this filter.
TIER_FILTER = "mid"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def _majority_run(rows: list[dict]) -> dict:
    """
    Collapse multiple run_id rows for one (task_id, provider) into a single
    classification + reasoning via majority vote across runs.
    Ties break to the first-seen row, ordered by run_id ascending.
    Reasoning is taken from the first row that produced the winning label.
    """
    rows_sorted = sorted(rows, key=lambda r: int(r.get("run_id") or 1))
    labels = [r["classification"].strip() for r in rows_sorted]
    counts = Counter(labels)
    top_count = max(counts.values())
    # first label (in run_id order) that has the top count
    winning_label = next(lbl for lbl in labels if counts[lbl] == top_count)
    winning_row = next(r for r in rows_sorted if r["classification"].strip() == winning_label)
    return {
        "classification": winning_label,
        "reasoning": winning_row["reasoning"].strip(),
    }


def load_long_format(filepath: str) -> dict[str, dict]:
    """
    Returns:
        {task_id: {
            "task_statement": str,
            "raters": {rater_key: {"classification": str, "reasoning": str}},
        }}
    Collapses repeated run_id rows per (task_id, provider) via majority vote,
    and filters model rows to TIER_FILTER (human rows always kept).
    """
    # raw[task_id][rater] = list of rows (across runs)
    raw: dict[str, dict[str, list[dict]]] = defaultdict(lambda: defaultdict(list))
    statements: dict[str, str] = {}

    with open(filepath, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            tid = row["task_id"].strip()
            provider = row["provider"].strip().lower()
            tier = (row.get("tier") or "").strip().lower()

            if provider not in PROVIDER_TO_RATER:
                continue  # unknown provider value, skip
            if provider != "human" and tier != TIER_FILTER:
                continue  # skip flagship / other tiers for model rows

            rater = PROVIDER_TO_RATER[provider]
            statements.setdefault(tid, row["task_statement"].strip())
            raw[tid][rater].append(row)

    tasks: dict[str, dict] = {}
    for tid, rater_rows in raw.items():
        tasks[tid] = {
            "task_statement": statements[tid],
            "raters": {
                rater: _majority_run(rows)
                for rater, rows in rater_rows.items()
            },
        }
    return tasks


def build_initial_tally(task_raters: dict[str, dict]) -> dict[str, list[str]]:
    tally: dict[str, list[str]] = defaultdict(list)
    for rater, data in task_raters.items():
        tally[data["classification"]].append(rater)
    return dict(tally)


def split_tasks(tasks: dict[str, dict]) -> tuple[dict, dict]:
    """
    auto_resolved — full 4/4 agreement only.
    disputed      — everything else.
    """
    auto_resolved: dict[str, dict] = {}
    disputed: dict[str, dict] = {}

    for tid, data in tasks.items():
        labels = [r["classification"] for r in data["raters"].values()]
        total = len(labels)
        tally = build_initial_tally(data["raters"])
        top_label, top_voters = max(tally.items(), key=lambda kv: len(kv[1]))
        top_count = len(top_voters)

        if top_count == total and total == len(ALL_RATERS):
            auto_resolved[tid] = {
                **data,
                "majority_label": top_label,
                "resolution_method": "initial_unanimous_4_4",
            }
        else:
            disputed[tid] = data

    return auto_resolved, disputed


def main() -> None:
    print(f"Loading initial round from: {INPUT_CSV}")
    tasks = load_long_format(INPUT_CSV)
    print(f"  {len(tasks)} tasks total")

    auto_resolved, disputed = split_tasks(tasks)
    print(
        f"\n  {len(auto_resolved)} auto-resolved (4/4 unanimous) → pass through\n"
        f"  {len(disputed)} disputed (anything less than 4/4) → consensus round"
    )

    # ── Write auto_resolved.csv ──────────────────────────────────────────
    auto_path = OUTPUT_DIR / "auto_resolved.csv"
    with open(auto_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "task_id", "task_statement", "final_label",
                "resolution_method", "vote_tally_initial",
            ],
        )
        writer.writeheader()
        for tid, data in sorted(auto_resolved.items()):
            writer.writerow({
                "task_id": tid,
                "task_statement": data["task_statement"],
                "final_label": data["majority_label"],
                "resolution_method": data["resolution_method"],
                "vote_tally_initial": json.dumps(build_initial_tally(data["raters"])),
            })

    # ── Write disputed_tasks.csv (long format, same shape as input) ─────
    disputed_path = OUTPUT_DIR / "disputed_tasks.csv"
    with open(disputed_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f, fieldnames=["task_id", "task_statement", "rater", "classification", "reasoning"]
        )
        writer.writeheader()
        for tid, data in sorted(disputed.items()):
            for rater in ALL_RATERS:
                if rater in data["raters"]:
                    r = data["raters"][rater]
                    writer.writerow({
                        "task_id": tid,
                        "task_statement": data["task_statement"],
                        "rater": rater,
                        "classification": r["classification"],
                        "reasoning": r["reasoning"],
                    })

    print(f"\n  Outputs:")
    print(f"    {auto_path}      — {len(auto_resolved)} tasks, pass through")
    print(f"    {disputed_path}  — {len(disputed)} tasks, needs consensus round")
    print(
        f"\n  NEXT STEP: read {disputed_path.name}, prepare human_consensus.csv "
        f"(task_id, human_consensus) for each disputed task, then run "
        f"02_consensus_round.py."
    )


if __name__ == "__main__":
    main()

Loading initial round from: june 23 unified_rater_long_format.csv
  212 tasks total

  131 auto-resolved (4/4 unanimous) → pass through
  81 disputed (anything less than 4/4) → consensus round

  Outputs:
    outputs/auto_resolved.csv      — 131 tasks, pass through
    outputs/disputed_tasks.csv  — 81 tasks, needs consensus round

  NEXT STEP: read disputed_tasks.csv, prepare human_consensus.csv (task_id, human_consensus) for each disputed task, then run 02_consensus_round.py.


In [2]:
import pandas as pd

disputed_df = pd.read_csv("outputs/disputed_tasks.csv")

for_human_consensus_df = disputed_df[['task_id', 'task_statement']].drop_duplicates().reset_index(drop=True)

for_human_consensus_df.to_csv("for_human_consensus.csv", index=False)

In [ ]:
"""
R2R Phase 1 — Step 2a: Consensus API Calls
===========================================
Calls Claude, OpenAI, and Gemini on each disputed task, showing each model
the OTHER raters' initial reasoning (labels withheld). Outputs one CSV of
raw model responses — no voting or resolution here.

Inputs:
  disputed_tasks.csv   — long format from 01_find_disputed.py
                         columns: task_id, task_statement, rater,
                                  classification, reasoning
                         rater values: human | claude | openai | gemini

Outputs (written to outputs/):
  consensus_prompts.csv  — audit trail: task_id, model, user_message
  consensus_raw.csv      — task_id, model, classification, reasoning,
                           input_tokens, output_tokens

Usage:
    python 02a_consensus_api_calls.py

Run modes (edit flags below, no CLI flags needed):
    DRY_RUN    = True  → preview first 3 prompts, no API calls
    SMOKE_TEST = True  → 3 API calls with synthetic data, then exit
"""

!pip install anthropic openai google-genai pydantic pandas

In [4]:
from __future__ import annotations

import csv
import json
import random
import time
from pathlib import Path

import anthropic
import openai
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

import os
os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["OPENAI_API_KEY"] = ""
os.environ["GEMINI_API_KEY"] = ""

In [5]:
# ---------------------------------------------------------------------------
# 0. RUN MODE
# ---------------------------------------------------------------------------

DISPUTED_CSV = "outputs/disputed_tasks.csv"

DRY_RUN    = False
SMOKE_TEST = False

# ---------------------------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------------------------

MODELS = {
    "claude": "claude-sonnet-4-6",
    "openai": "gpt-5.4-mini",
    "gemini": "gemini-3.1-flash-lite",
}

ALL_INITIAL_RATERS = ["human", "claude", "openai", "gemini"]

ALLOWED_LABELS = [
    "Master Data Management",
    "Record",
    "Close & Consolidation",
    "Report",
    "Analysis & Business Insight",
    "Planning & Budgeting",
    "OUT_OF_SCOPE",
]

OUTPUT_DIR = Path("consensus outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

SLEEP_BETWEEN_CALLS = 0.5  # seconds; increase if hitting rate limits

# ---------------------------------------------------------------------------
# 2. SUBPROCESS DEFINITIONS AND DECISION RULES
# ---------------------------------------------------------------------------

SUBPROCESS_DEFINITIONS = """
1. Master Data Management — Establishing and maintaining the structural and 
   governance foundation of the financial architecture: the chart of accounts, 
   reporting entity hierarchies, cost and profit centre structures, 
   company-specific attributes, currency rate tables, system interfaces, and user 
   access rights. Provides the infrastructure on which all downstream subprocesses 
   depend, without itself processing period transactions.

2. Record — The ongoing accumulation and initial processing of financial 
   transactions during the fiscal period, per accepted accounting principles: 
   processing journal entries, posting transactions to subledgers, recording accruals, 
   and performing transaction-level reconciliations as activity occurs.

3. Close & Consolidation — The period-end procedures that finalise and aggregate 
   recorded transactions into group-level financial results: closing subledgers, executing 
   general ledger accounting, finalising accruals against the close deadline, reconciling GL 
   accounts and resolving variances, managing intercompany accounting and eliminations, 
   calculating tax liability (different from tax filings), and consolidating entity results into 
   preliminary group financial statements.

4. Report — Translating the closed and consolidated ledger into structured outputs 
   for stakeholders: producing internal financial management reports, external 
   financial statements, and tax filings, in order to meet the organisation's 
   internal and external disclosure obligations.

5. Analysis & Business Insight — Interrogating reported financial results to 
   generate interpretive, decision-useful intelligence: performing variance and 
   performance analysis, developing and maintaining KPIs, and providing insight 
   intended to influence management decisions, including post-close review and 
   identification of needed adjustments.

6. Planning & Budgeting — Translating analytical findings into forward-looking 
   financial direction: developing long-range strategic plans, annual budgets and 
   operating plans, rolling forecasts, product costing, and performance targets 
   that govern the next iteration of the cycle.
"""

DECISION_RULES = """
Apply Rules 0 through 4 strictly in numerical order, one rule at a time. Fully
resolve or fully rule out each rule before moving to the next — do not jump
ahead to a later rule to justify a classification that an earlier rule would
have already determined differently. Begin every task with Rule 0.

Rule 0 — Before applying this rule, read the subprocess definitions carefully. 
  The scope of R2R in this study is determined exclusively by those definitions, 
  not simply general intuitions about what constitutes "financial" work. Do not exclude 
  a task on the grounds that it resembles operational, engineering, or non-accounting 
  work if the subprocess definitions would encompass it. A task is in scope if it constitutes 
  activity within any of the six subprocess definitions provided. If the task instead 
  consists of data-gathering, or cross-functional communication that precedes and informs 
  subprocess activity — but does not itself constitute such activity according to the defnition 
  — classify as OUT_OF_SCOPE.
  
  Exception — Master Data Management carve-in: A task that maintains,
  establishes, or governs a structural element of the financial architecture
  (chart of accounts, entity hierarchy, cost/profit centres, currency tables,
  system access) IS constitutive R2R activity, even though it does not itself
  process a transaction, figure, report, or plan. Treat it as in-scope and
  proceed to Rule 3.

  Carve-out — Audit/assurance activity: A task describing audit, assurance, or
  internal control work — testing controls, verifying compliance, examining
  records or processes for accuracy as an independent check, assessing the
  adequacy of a control environment — is OUT_OF_SCOPE, even when it involves
  reviewing financial records or transactions. Its purpose is independent
  verification of work already performed, not the performance of the
  recording, consolidation, disclosure, analysis, or planning itself. (O*NET
  groups "Accountants and Auditors" under shared occupational codes; this
  carve-out separates the two distinct job functions.)

  Carve-out — Tax planning/strategy: A task describing tax planning, tax
  strategy, or general tax advisory (e.g., developing tax-minimisation
  strategies, advising on entity tax structuring) is OUT_OF_SCOPE. It does not
  itself produce a recorded entry, consolidated tax provision, filed
  disclosure, or budget. Distinguish from tax provision calculation (→ Close &
  Consolidation) and tax filing/return preparation (→ Report), which remain
  in scope. Distinguish from other financial planning and strategic activities, which
  remain in scope.

Rule 1 — Purpose of output (primary rule).
  Classify by why the output is being produced, not by the activity that
  produces it. A task that prepares reports for management analysis is
  Analysis & Business Insight, not Report, because the purpose is decision
  support, not disclosure.

  Note: This rule routes a task to Analysis & Business Insight only when the
  analysis interrogates the organisation's own recorded financial or
  accounting data (e.g., variance to budget, financial KPI trend, post-close
  review). Analysis of external information — market research, competitor
  benchmarking, macroeconomic or industry forecasting — does not qualify, even
  if the task is described as "financial analysis"; such tasks fail Rule 0 and
  are OUT_OF_SCOPE.

Rule 2 — Output type (fallback when Rule 1 leaves no clear purpose).
  Classify by the type of artefact produced:
    - A recorded or reconciled entry → Record
    - A consolidated group figure → Close & Consolidation
    - A disclosure document or regulatory filing → Report
    - A budget, forecast, or performance target → Planning & Budgeting

Rule 3 — Maintenance vs. execution (Master Data Management boundary).
  A task framed as maintaining, establishing, or governing a structural
  element (chart of accounts, entity hierarchy, cost/profit centres, currency
  tables, system access) → Master Data Management, regardless of which
  subprocess uses that structure. A task that uses such structure to process
  a period's transactions → the relevant operational subprocess per Rules 1–2.

Rule 4 — Transaction-level vs. period-end (Record vs. Close &
Consolidation).
  Is the task processing an individual transaction, or closing out a period?
  Transaction-level reconciliations and adjustments → Record.
  Account-balance or period-end reconciliations, adjustments, and reviews →
  Close & Consolidation.

Rule 5 — Single-label enforcement.
  Apply Rules 0–4 in sequence to arrive at exactly one label, even where a
  task description spans multiple subprocesses. Multi-subprocess phrasing
  alone is not grounds for AMBIGUOUS. Reserve AMBIGUOUS only for genuine
  unresolved contestation after all rules are exhausted, where both purpose and output 
  rules cannot resolve which subprocess to put the task in.
"""

# ---------------------------------------------------------------------------
# 3. PROMPT BUILDERS
# ---------------------------------------------------------------------------

CONSENSUS_SYSTEM_PROMPT = f"""You are an expert in Record-to-Report (R2R) process classification, \
participating in a deliberative consensus round.

In an earlier independent round, multiple raters classified the same O*NET task statements \
into subprocesses of the R2R accounting process. You will be shown the reasoning from those \
other raters — but not their labels — for tasks where the initial round produced a split.

Your task is to read each argument carefully, weigh its merits against the subprocess \
definitions and decision rules below, and provide your own independent classification.

SUBPROCESS DEFINITIONS:
{SUBPROCESS_DEFINITIONS}

DECISION RULES:
{DECISION_RULES}

ALLOWED LABELS — use exactly as written:
{chr(10).join(f'  - {label}' for label in ALLOWED_LABELS)}

IMPORTANT INSTRUCTIONS:
- AMBIGUOUS is NOT valid at this stage. You must commit to one of the six subprocesses \
or OUT_OF_SCOPE. The purpose of this round is to resolve uncertainty, not carry it forward.
- Focus on the quality of the arguments, not the number of raters who held a position.
- Do not anchor to your own prior classification. Re-examine the task fresh against the rules.
- Apply the rules in order (Rule 0 first); stop at the first rule that resolves the case.

RESPONSE FORMAT:
Respond with a valid JSON object only — no other text before or after.

{{
  "reasoning": "<1–2 sentences: engage with the other raters' arguments, name the rule \
that resolves the case, state why>",
  "classification": "<exactly one of the allowed labels>"
}}"""


def build_consensus_user_message(
    task_id: str,
    task_statement: str,
    other_reasonings: list[dict],
) -> str:
    parts = [
        f"Task ID: {task_id}",
        f"Task Statement: {task_statement}",
        "",
        "--- Reasoning from other raters (anonymised, labels withheld) ---",
        "",
    ]
    for entry in other_reasonings:
        parts.append(f"{entry['anon_label']}:")
        parts.append(entry["reasoning"].strip())
        parts.append("")
    parts.append("---")
    parts.append("")
    parts.append(
        "Having read the arguments above, provide your own independent classification "
        "of this task. Remember: AMBIGUOUS is not valid at this stage."
    )
    return "\n".join(parts)


def build_other_reasonings(
    task_raters: dict[str, dict],
    current_model: str,
    seed: int | None = None,
) -> list[dict]:
    others = [
        r for r in ALL_INITIAL_RATERS
        if r != current_model and r in task_raters
    ]
    rng = random.Random(seed)
    rng.shuffle(others)
    anon = [chr(65 + i) for i in range(len(others))]
    return [
        {
            "anon_label": f"Rater {anon[i]}",
            "reasoning": task_raters[others[i]]["reasoning"],
        }
        for i in range(len(others))
    ]

# ---------------------------------------------------------------------------
# 4. PYDANTIC SCHEMA
# ---------------------------------------------------------------------------

class ConsensusClassification(BaseModel):
    reasoning: str = Field(description="1-2 sentences engaging with other raters' arguments.")
    classification: str = Field(description=f"One of: {', '.join(ALLOWED_LABELS)}")

# ---------------------------------------------------------------------------
# 5. API CALLERS
# ---------------------------------------------------------------------------

def _parse_json(raw: str) -> dict:
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"classification": "PARSE_ERROR", "reasoning": raw}


def call_claude(task_id: str, task_statement: str, other_reasonings: list[dict]) -> dict:
    client = anthropic.Anthropic()
    user_msg = build_consensus_user_message(task_id, task_statement, other_reasonings)
    response = client.messages.create(
        model=MODELS["claude"],
        max_tokens=512,
        system=CONSENSUS_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_msg}],
    )
    parsed = _parse_json(response.content[0].text.strip())
    return {
        **parsed,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "user_message": user_msg,
    }


def call_openai(task_id: str, task_statement: str, other_reasonings: list[dict]) -> dict:
    client = openai.OpenAI()
    user_msg = build_consensus_user_message(task_id, task_statement, other_reasonings)
    response = client.beta.chat.completions.parse(
        model=MODELS["openai"],
        messages=[
            {"role": "system", "content": CONSENSUS_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        response_format=ConsensusClassification,
    )
    result = response.choices[0].message.parsed
    return {
        "classification": result.classification,
        "reasoning": result.reasoning,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
        "user_message": user_msg,
    }


def call_gemini(task_id: str, task_statement: str, other_reasonings: list[dict]) -> dict:
    client = genai.Client()
    user_msg = build_consensus_user_message(task_id, task_statement, other_reasonings)
    response = client.models.generate_content(
        model=MODELS["gemini"],
        contents=user_msg,
        config=types.GenerateContentConfig(
            system_instruction=CONSENSUS_SYSTEM_PROMPT,
            response_mime_type="application/json",
            response_schema=ConsensusClassification,
        ),
    )
    parsed = json.loads(response.text)
    return {
        "classification": parsed.get("classification", "PARSE_ERROR"),
        "reasoning": parsed.get("reasoning", ""),
        "input_tokens": response.usage_metadata.prompt_token_count,
        "output_tokens": response.usage_metadata.candidates_token_count,
        "user_message": user_msg,
    }


PROVIDER_FNS = {
    "claude": call_claude,
    "openai": call_openai,
    "gemini": call_gemini,
}

# ---------------------------------------------------------------------------
# 6. DATA LOADER
# ---------------------------------------------------------------------------

def load_disputed_tasks(filepath: str) -> dict[str, dict]:
    tasks: dict[str, dict] = {}
    with open(filepath, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            tid = row["task_id"].strip()
            if tid not in tasks:
                tasks[tid] = {
                    "task_statement": row["task_statement"].strip(),
                    "raters": {},
                }
            tasks[tid]["raters"][row["rater"].strip()] = {
                "classification": row["classification"].strip(),
                "reasoning": row["reasoning"].strip(),
            }
    return tasks

# ---------------------------------------------------------------------------
# 7. SMOKE TEST
# ---------------------------------------------------------------------------

def smoke_test() -> None:
    task_id = "3394"
    task_statement = (
        "Determine depreciation rates to apply to capitalized items and advise "
        "management on actions regarding the purchase, lease, or disposal of such items."
    )
    synthetic = [
        {
            "anon_label": "Rater A",
            "reasoning": (
                "The task produces a depreciation rate — a calculated figure applied "
                "to recorded asset values. Rule 2 points to Record since the output "
                "feeds directly into the journal entry for depreciation expense."
            ),
        },
        {
            "anon_label": "Rater B",
            "reasoning": (
                "The advisory component — recommending purchase, lease, or disposal actions "
                "to management — is the primary purpose. Rule 1: the output is decision-useful "
                "intelligence, pointing to Analysis & Business Insight."
            ),
        },
        {
            "anon_label": "Rater C",
            "reasoning": (
                "Depreciation rate determination governs how an asset is recorded over time. "
                "Rule 3 clarifies MDM covers structural elements like chart of accounts, not "
                "rate policies — so I lean toward Record."
            ),
        },
    ]
    print("=" * 60)
    print("SMOKE TEST — 3 API calls, one per provider")
    print(f"Task {task_id}: {task_statement[:70]}...")
    print("=" * 60)
    for provider, call_fn in PROVIDER_FNS.items():
        print(f"\n[{provider.upper()}]  {MODELS[provider]}")
        try:
            result = call_fn(task_id, task_statement, synthetic)
            print(f"  Classification : {result['classification']}")
            print(f"  Reasoning      : {result['reasoning']}")
            print(f"  Tokens         : {result['input_tokens']} in / {result['output_tokens']} out")
        except Exception as e:
            print(f"  ERROR: {e}")

# ---------------------------------------------------------------------------
# 8. MAIN
# ---------------------------------------------------------------------------

def main() -> None:
    if SMOKE_TEST:
        smoke_test()
        return

    print(f"Loading disputed tasks from: {DISPUTED_CSV}")
    disputed = load_disputed_tasks(DISPUTED_CSV)
    print(f"  {len(disputed)} disputed tasks loaded")

    if not disputed:
        print("No disputed tasks — nothing to do.")
        return

    prompts_path = OUTPUT_DIR / "consensus_prompts.csv"
    raw_path     = OUTPUT_DIR / "consensus_raw.csv"

    prompts_fields = ["task_id", "model", "user_message"]
    raw_fields     = [
        "task_id", "task_statement", "model",
        "classification", "reasoning",
        "input_tokens", "output_tokens",
    ]

    if DRY_RUN:
        print("\n=== DRY RUN — first 3 prompts, no API calls ===\n")
        for tid, data in list(disputed.items())[:3]:
            print(f"Task {tid}: {data['task_statement'][:80]}...")
            for model in MODELS:
                others = build_other_reasonings(
                    data["raters"], model,
                    seed=hash((tid, model)) % 2**32,
                )
                msg = build_consensus_user_message(tid, data["task_statement"], others)
                print(f"  [{model}] {len(msg)} chars | first 150: {msg[:150].replace(chr(10),' ')}")
            print()
        return

    total = len(disputed) * len(MODELS)
    done  = 0

    with (
        open(prompts_path, "w", newline="", encoding="utf-8") as pf,
        open(raw_path,     "w", newline="", encoding="utf-8") as rf,
    ):
        pw = csv.DictWriter(pf, fieldnames=prompts_fields)
        rw = csv.DictWriter(rf, fieldnames=raw_fields)
        pw.writeheader()
        rw.writeheader()

        for tid, data in disputed.items():
            for model, call_fn in PROVIDER_FNS.items():
                done += 1
                print(f"[{done}/{total}]  {model} | task {tid}")

                others = build_other_reasonings(
                    data["raters"], model,
                    seed=hash((tid, model)) % 2**32,
                )

                try:
                    result = call_fn(tid, data["task_statement"], others)
                except Exception as e:
                    print(f"  ERROR: {e}")
                    result = {
                        "classification": "API_ERROR",
                        "reasoning": str(e),
                        "input_tokens": 0,
                        "output_tokens": 0,
                        "user_message": "",
                    }

                pw.writerow({
                    "task_id": tid,
                    "model": model,
                    "user_message": result.get("user_message", ""),
                })
                rw.writerow({
                    "task_id": tid,
                    "task_statement": data["task_statement"],
                    "model": model,
                    "classification": result["classification"],
                    "reasoning": result["reasoning"],
                    "input_tokens": result["input_tokens"],
                    "output_tokens": result["output_tokens"],
                })
                pf.flush()
                rf.flush()
                time.sleep(SLEEP_BETWEEN_CALLS)

    print(f"\n=== API calls complete ===")
    print(f"  {prompts_path}  — prompt audit trail")
    print(f"  {raw_path}      — model consensus classifications")
    print(f"\nNext step: run 02b_tally_and_vote.py")

In [6]:
if __name__ == "__main__":
    main()

Loading disputed tasks from: outputs/disputed_tasks.csv
  81 disputed tasks loaded
[1/243]  claude | task 1299
[2/243]  openai | task 1299
[3/243]  gemini | task 1299
[4/243]  claude | task 1300
[5/243]  openai | task 1300
[6/243]  gemini | task 1300
[7/243]  claude | task 1306
[8/243]  openai | task 1306
[9/243]  gemini | task 1306
[10/243]  claude | task 1309
[11/243]  openai | task 1309
[12/243]  gemini | task 1309
[13/243]  claude | task 1311
[14/243]  openai | task 1311
[15/243]  gemini | task 1311
[16/243]  claude | task 1313
[17/243]  openai | task 1313
[18/243]  gemini | task 1313
[19/243]  claude | task 18561
[20/243]  openai | task 18561
[21/243]  gemini | task 18561
[22/243]  claude | task 20846
[23/243]  openai | task 20846
[24/243]  gemini | task 20846
[25/243]  claude | task 21319
[26/243]  openai | task 21319
[27/243]  gemini | task 21319
[28/243]  claude | task 21321
[29/243]  openai | task 21321
[30/243]  gemini | task 21321
[31/243]  claude | task 21322
[32/243]  open

In [7]:
"""
R2R Phase 1 — Step 2b: Merge Consensus Votes
=============================================
Merges your human consensus Excel file (numeric subprocess codes) with the
model consensus raw CSV from 02a into a single long-format file.

Does NOT do any voting or resolution — that happens in 02c_tally_and_vote.py.

Inputs:
  outputs/consensus_raw.csv        — from 02a_consensus_api_calls.py
                                     columns: task_id, task_statement, model,
                                              classification, reasoning,
                                              input_tokens, output_tokens

  june21 human consensus.xlsx      — your manual consensus file with numeric
                                     subprocess codes (0–7)

Outputs (written to outputs/):
  consensus_all_votes_long.csv     — one row per (task_id, rater) across the
                                     full consensus round (human + 3 models)
                                     columns: task_id, task_statement, rater,
                                              classification, reasoning

Usage:
    python 02b_merge_consensus.py

Configuration:
  Edit the block below to match your Excel column names.
  The subprocess_map must match your numeric coding scheme exactly.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# CONFIGURATION — edit to match your Excel file
# ---------------------------------------------------------------------------

CONSENSUS_RAW_CSV    = "consensus outputs/consensus_raw.csv"
HUMAN_CONSENSUS_XLSX = "june22_human_consensus.xlsx"
HUMAN_XLSX_SHEET     = 0           # 0 = first sheet, or use sheet name as string

# Column names inside the Excel (strip-matched, so extra spaces are fine)
HUMAN_XLSX_TASK_ID_COL   = "task_id"        # numeric task ID column
HUMAN_XLSX_LABEL_COL     = "classification"  # numeric subprocess code column (0–7)
HUMAN_XLSX_REASONING_COL = "reasoning"       # reasoning column; set None if absent

# Numeric code → text label mapping (must match what models output exactly)
SUBPROCESS_MAP = {
    0: "OUT_OF_SCOPE",
    1: "Master Data Management",
    2: "Record",
    3: "Close & Consolidation",
    4: "Report",
    5: "Analysis & Business Insight",
    6: "Planning & Budgeting",
    7: "AMBIGUOUS",
}

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# ---------------------------------------------------------------------------
# LOADERS
# ---------------------------------------------------------------------------

def load_model_consensus(filepath: str) -> pd.DataFrame:
    """
    Load consensus_raw.csv from 02a.
    Renames 'model' → 'rater' to unify column naming with human rows.
    """
    df = pd.read_csv(filepath, dtype=str).fillna("")
    missing = {"task_id", "task_statement", "model", "classification", "reasoning"} - set(df.columns)
    if missing:
        raise ValueError(f"consensus_raw.csv is missing columns: {missing}")

    df = df.rename(columns={"model": "rater"})
    df["task_id"] = df["task_id"].str.strip()
    return df[["task_id", "task_statement", "rater", "classification", "reasoning"]]


def load_human_consensus(
    filepath: str,
    sheet,
    task_id_col: str,
    label_col: str,
    reasoning_col: str | None,
) -> pd.DataFrame:
    """
    Load the human consensus Excel, recode numeric subprocess codes → text labels,
    and return a long-format DataFrame with the same schema as model rows.
    """
    raw = pd.read_excel(filepath, sheet_name=sheet, dtype=str).fillna("")
    raw.columns = [c.strip() for c in raw.columns]          # strip header whitespace

    # Validate columns
    for col in [task_id_col, label_col]:
        if col not in raw.columns:
            raise ValueError(
                f"Column '{col}' not found in {filepath}.\n"
                f"Available columns: {list(raw.columns)}"
            )

    # --- task_id: parse as int, then back to string to match model CSV ---
    task_ids_raw = pd.to_numeric(raw[task_id_col].str.strip(), errors="coerce")
    n_bad_ids = task_ids_raw.isna().sum()
    if n_bad_ids:
        print(f"  WARNING: {n_bad_ids} unparseable task_id(s) → dropped.")
    raw = raw[task_ids_raw.notna()].copy()
    task_ids_raw = task_ids_raw.dropna()
    task_id_str = task_ids_raw.astype(int).astype(str)

    # --- classification: numeric code → text label ---
    numeric_codes = pd.to_numeric(raw[label_col].str.strip(), errors="coerce")
    n_bad_codes = numeric_codes.isna().sum()
    if n_bad_codes:
        print(f"  WARNING: {n_bad_codes} unparseable subprocess code(s) → will be NaN.")

    text_labels = numeric_codes.map(SUBPROCESS_MAP)
    n_unmapped = text_labels.isna().sum()
    if n_unmapped:
        bad_vals = numeric_codes[text_labels.isna()].unique()
        print(
            f"  WARNING: {n_unmapped} code(s) not in SUBPROCESS_MAP → will be blank.\n"
            f"  Unrecognised values: {bad_vals}"
        )

    # --- reasoning ---
    if reasoning_col and reasoning_col in raw.columns:
        reasoning = raw[reasoning_col].str.strip()
    else:
        reasoning = pd.Series([""] * len(raw), index=raw.index)

    out = pd.DataFrame({
        "task_id":        task_id_str.values,
        "task_statement": "",           # filled from model CSV below
        "rater":          "human",
        "classification": text_labels.values,
        "reasoning":      reasoning.values,
    })

    return out


# ---------------------------------------------------------------------------
# MERGE
# ---------------------------------------------------------------------------

def merge_votes(model_df: pd.DataFrame, human_df: pd.DataFrame) -> pd.DataFrame:
    """
    Back-fill task_statement from model rows onto human rows, then stack.
    """
    stmt_map = (
        model_df[["task_id", "task_statement"]]
        .drop_duplicates("task_id")
        .set_index("task_id")["task_statement"]
        .to_dict()
    )
    human_df = human_df.copy()
    human_df["task_statement"] = human_df["task_id"].map(stmt_map).fillna("")

    combined = pd.concat([model_df, human_df], ignore_index=True)
    combined["classification"] = combined["classification"].fillna("").str.strip()
    combined = combined.sort_values(["task_id", "rater"]).reset_index(drop=True)
    return combined


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

def main() -> None:
    # --- Load model consensus ---
    print(f"Loading model consensus from: {CONSENSUS_RAW_CSV}")
    model_df = load_model_consensus(CONSENSUS_RAW_CSV)
    print(f"  {len(model_df)} rows | {model_df['task_id'].nunique()} unique tasks")
    print(f"  Raters found: {sorted(model_df['rater'].unique())}")

    # --- Load human consensus ---
    print(f"\nLoading human consensus from: {HUMAN_CONSENSUS_XLSX}  (sheet={HUMAN_XLSX_SHEET})")
    human_df = load_human_consensus(
        HUMAN_CONSENSUS_XLSX,
        sheet=HUMAN_XLSX_SHEET,
        task_id_col=HUMAN_XLSX_TASK_ID_COL,
        label_col=HUMAN_XLSX_LABEL_COL,
        reasoning_col=HUMAN_XLSX_REASONING_COL,
    )
    print(f"  {len(human_df)} rows after recoding")

    # Spot-check recoding
    print("\n  Recoding spot-check (first 5 human rows):")
    print(human_df[["task_id", "classification", "reasoning"]].head().to_string(index=False))

    # --- Alignment check ---
    model_tasks = set(model_df["task_id"].unique())
    human_tasks = set(human_df["task_id"].unique())
    only_model  = model_tasks - human_tasks
    only_human  = human_tasks - model_tasks

    if only_model:
        print(
            f"\n  WARNING: {len(only_model)} task(s) in model CSV have no human entry "
            f"— they will show up in the tally with a missing human vote.\n"
            f"  IDs: {sorted(only_model, key=int)[:15]}"
            f"{'...' if len(only_model) > 15 else ''}"
        )
    if only_human:
        print(
            f"\n  WARNING: {len(only_human)} task(s) in human Excel have no model rows "
            f"— check task_id alignment.\n"
            f"  IDs: {sorted(only_human, key=int)[:15]}"
            f"{'...' if len(only_human) > 15 else ''}"
        )

    # --- Merge ---
    long_df = merge_votes(model_df, human_df)

    out_path = OUTPUT_DIR / "consensus_all_votes_long.csv"
    long_df.to_csv(out_path, index=False)

    print(f"\n=== Merge complete ===")
    print(f"  Rows in merged file: {len(long_df)}")
    print(f"  Unique tasks:        {long_df['task_id'].nunique()}")
    print(f"  Raters:              {sorted(long_df['rater'].unique())}")
    print(f"\n  Saved: {out_path}")
    print(f"\nNext step: run 02c_tally_and_vote.py")


if __name__ == "__main__":
    main()

Loading model consensus from: consensus outputs/consensus_raw.csv
  243 rows | 81 unique tasks
  Raters found: ['claude', 'gemini', 'openai']

Loading human consensus from: june22_human_consensus.xlsx  (sheet=0)
  81 rows after recoding

  Recoding spot-check (first 5 human rows):
task_id         classification                                                                                                                                                                   reasoning
   1299           OUT_OF_SCOPE                                                                                                                               a bit more generic than specific for R2R work
   1300           OUT_OF_SCOPE                                                                                                                               a bit more generic than specific for R2R work
   1306 Master Data Management                                                         Rule 0 carve-in, data 

In [8]:
"""
R2R Phase 1 — Step 2c: Tally and Vote
=======================================
Reads the merged long-format consensus votes from 02b and resolves a final
label for each disputed task by majority rule across the four consensus-round
raters (human + claude + openai + gemini).

Does NOT merge files — that is handled by 02b_merge_consensus.py.

Inputs:
  outputs/consensus_all_votes_long.csv  — from 02b_merge_consensus.py
                                          columns: task_id, task_statement,
                                                   rater, classification,
                                                   reasoning

Outputs (written to outputs/):
  consensus_resolved.csv               — one row per task_id:
                                           task_id, task_statement,
                                           final_label, resolution_method,
                                           vote_tally, needs_human_review,
                                           human_consensus,
                                           claude_consensus,
                                           openai_consensus,
                                           gemini_consensus

Resolution logic (up to 4 voters: human + claude + openai + gemini):
  ≥ 3 agree on same label  →  resolved (majority or unanimous)
  2/2 split                →  final_label blank, needs_human_review = True
  voter absent             →  counted as missing, not as a vote

Usage:
    python 02c_tally_and_vote.py

After running:
  Open outputs/consensus_resolved.csv, filter needs_human_review = True,
  manually set final_label, then run 02d_build_final_register.py.
"""

from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------------------------

LONG_CSV   = "outputs/consensus_all_votes_long.csv"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Expected raters in the consensus round
CONSENSUS_RATERS = ["human", "claude", "openai", "gemini"]

# Labels that are valid for a resolved final_label
# AMBIGUOUS is listed here because a human may have entered it —
# it will resolve normally if three raters agree on it, but it will
# be flagged in the summary so you can decide how to handle it.
VALID_LABELS = {
    "Master Data Management",
    "Record",
    "Close & Consolidation",
    "Report",
    "Analysis & Business Insight",
    "Planning & Budgeting",
    "OUT_OF_SCOPE",
    "AMBIGUOUS",   # included so it doesn't silently distort the tally
}

# ---------------------------------------------------------------------------
# TALLY LOGIC
# ---------------------------------------------------------------------------

def resolve_votes(long_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each task_id, collect consensus-round votes and determine final label.

    Returns one row per task_id.
    """
    rows = []

    for tid, grp in long_df.groupby("task_id", sort=False):
        task_statement = grp["task_statement"].iloc[0]

        # Build {rater: classification} — last row wins if duplicate raters exist
        votes: dict[str, str] = {}
        for _, r in grp.iterrows():
            rater = r["rater"].strip()
            label = r["classification"].strip()
            if label:                   # ignore blank/missing entries
                votes[rater] = label

        # Warn about blank classifications
        blank_raters = [
            r["rater"] for _, r in grp.iterrows()
            if not r["classification"].strip()
        ]
        if blank_raters:
            print(
                f"  INFO task {tid}: blank classification for rater(s) "
                f"{blank_raters} — not counted in vote."
            )

        # Count votes
        tally: dict[str, list[str]] = defaultdict(list)
        for rater, label in votes.items():
            tally[label].append(rater)

        sorted_tally = sorted(tally.items(), key=lambda x: len(x[1]), reverse=True)
        top_label, top_voters = sorted_tally[0]
        top_count   = len(top_voters)
        total_votes = len(votes)

        if top_count >= 3:
            method = (
                "consensus_unanimous"
                if top_count == total_votes
                else "consensus_majority_3plus"
            )
            final_label        = top_label
            needs_human_review = False
        else:
            # True 2/2 split, or a 2-way tie with fewer than 3 voters present
            method             = "2_2_split"
            final_label        = ""
            needs_human_review = True

        rows.append({
            "task_id":            str(tid),
            "task_statement":     task_statement,
            "final_label":        final_label,
            "resolution_method":  method,
            "vote_tally":         json.dumps(dict(tally)),
            "needs_human_review": needs_human_review,
            "human_consensus":    votes.get("human", ""),
            "claude_consensus":   votes.get("claude", ""),
            "openai_consensus":   votes.get("openai", ""),
            "gemini_consensus":   votes.get("gemini", ""),
        })

    df = pd.DataFrame(rows)
    # Sort numerically if task_ids are numeric strings
    try:
        df = df.sort_values("task_id", key=lambda s: s.astype(int)).reset_index(drop=True)
    except (ValueError, TypeError):
        df = df.sort_values("task_id").reset_index(drop=True)

    return df


# ---------------------------------------------------------------------------
# DIAGNOSTICS
# ---------------------------------------------------------------------------

def print_summary(resolved_df: pd.DataFrame) -> None:
    n_total      = len(resolved_df)
    n_resolved   = (resolved_df["needs_human_review"] == False).sum()
    n_unresolved = (resolved_df["needs_human_review"] == True).sum()

    # Resolution method breakdown
    method_counts = resolved_df["resolution_method"].value_counts()

    # Flag any AMBIGUOUS that made it through as a final label
    n_ambiguous = (resolved_df["final_label"] == "AMBIGUOUS").sum()

    print(f"\n=== Tally complete ===")
    print(f"  Total tasks:                   {n_total}")
    print(f"    → Resolved (≥3/4 agreement): {n_resolved}")
    print(f"    → Needs human review (2/2):  {n_unresolved}")
    print(f"\n  Resolution methods:")
    for method, count in method_counts.items():
        print(f"    {method:<35} {count}")

    if n_ambiguous:
        print(
            f"\n  NOTE: {n_ambiguous} task(s) resolved to AMBIGUOUS.\n"
            f"  These will propagate to the final register. Consider whether\n"
            f"  they should be manually reclassified or treated as OUT_OF_SCOPE."
        )

    # Resolved label distribution (excluding needs_human_review)
    resolved_only = resolved_df[resolved_df["needs_human_review"] == False]
    if not resolved_only.empty:
        print(f"\n  Label distribution (resolved tasks):")
        label_counts = resolved_only["final_label"].value_counts()
        for label, count in label_counts.items():
            print(f"    {label:<35} {count}")

    if n_unresolved:
        unresolved_ids = resolved_df.loc[
            resolved_df["needs_human_review"] == True, "task_id"
        ].tolist()
        print(f"\n  ACTION REQUIRED:")
        print(f"  Open outputs/consensus_resolved.csv, filter needs_human_review = True,")
        print(f"  and manually set final_label for each task listed below.")
        print(f"  Task IDs: {unresolved_ids}")


# ---------------------------------------------------------------------------
# MAIN
# ---------------------------------------------------------------------------

def main() -> None:
    print(f"Loading merged votes from: {LONG_CSV}")
    long_df = pd.read_csv(LONG_CSV, dtype=str).fillna("")
    print(f"  {len(long_df)} rows | {long_df['task_id'].nunique()} unique tasks")

    missing_cols = {"task_id", "task_statement", "rater", "classification"} - set(long_df.columns)
    if missing_cols:
        raise ValueError(f"consensus_all_votes_long.csv is missing columns: {missing_cols}")

    # Report which raters are actually present
    raters_found = sorted(long_df["rater"].unique())
    raters_missing = [r for r in CONSENSUS_RATERS if r not in raters_found]
    print(f"  Raters found:   {raters_found}")
    if raters_missing:
        print(f"  WARNING — expected raters not in file: {raters_missing}")

    # --- Tally ---
    print("\nResolving votes...")
    resolved_df = resolve_votes(long_df)

    # --- Save ---
    out_path = OUTPUT_DIR / "consensus_resolved.csv"
    resolved_df.to_csv(out_path, index=False)
    print(f"\n  Saved: {out_path}")

    # --- Summary ---
    print_summary(resolved_df)

    print(f"\nNext step:")
    print(f"  If needs_human_review tasks exist → fill final_label manually, then")
    print(f"  run 02d_build_final_register.py")
    print(f"  If all resolved → run 02d_build_final_register.py directly.")


if __name__ == "__main__":
    main()

Loading merged votes from: outputs/consensus_all_votes_long.csv
  324 rows | 81 unique tasks
  Raters found:   ['claude', 'gemini', 'human', 'openai']

Resolving votes...

  Saved: outputs/consensus_resolved.csv

=== Tally complete ===
  Total tasks:                   81
    → Resolved (≥3/4 agreement): 73
    → Needs human review (2/2):  8

  Resolution methods:
    consensus_majority_3plus            39
    consensus_unanimous                 34
    2_2_split                           8

  Label distribution (resolved tasks):
    OUT_OF_SCOPE                        31
    Record                              15
    Planning & Budgeting                14
    Report                              6
    Analysis & Business Insight         3
    Master Data Management              2
    Close & Consolidation               2

  ACTION REQUIRED:
  Open outputs/consensus_resolved.csv, filter needs_human_review = True,
  and manually set final_label for each task listed below.
  Task IDs: ['249